##Imports

In [0]:
from pyspark.sql.functions import (
    cast, coalesce, col, concat_ws, count, current_date, 
    current_timestamp, lit, sha2
)
from pyspark.sql.types import LongType, StringType, StructField, StructType
import uuid

## Configuration

In [0]:
SOURCE_PATH = "/Volumes/workspace/final/final_project"

CATALOG = "workspace"
SCHEMA = "final"

BATCH_ID = str(uuid.uuid4())

print("Batch ID :", BATCH_ID)

## File Discovery

In [0]:
files = dbutils.fs.ls(SOURCE_PATH)

for file in files:
    print(file.name)

## Schema Setup

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.final
""")

## Helper Functions

In [0]:
def read_source_file(file):

    file_name = file.name
    file_path = file.path

    print("=" * 50)
    print(f"Reading : {file_name}")

    if file_name.endswith(".csv"):

        if file_name == "Person.csv":
            df = (
                spark.read
                .option("header", True)
                .option("delimiter", "|")
                .csv(file_path)
            )

        else:
            df = (
                spark.read
                .option("header", True)
                .csv(file_path)
            )

    elif file_name.endswith(".json"):
        df = spark.read.json(file_path)
        
    else:
        raise Exception(f"Unsupported file : {file_name}")

    print("Columns:")
    print(df.columns)

    return df

In [0]:
def add_bronze_metadata(df, file_name):

    return (
        df
        .withColumn(
            "source_file_name",
            lit(file_name)
        )
        .withColumn(
            "ingested_at",
            current_timestamp()
        )
        .withColumn(
            "batch_id",
            lit(BATCH_ID)
        )
        .withColumn(
            "load_date",
            current_date()
        )
        .withColumn(
            "record_hash",
            sha2(
                concat_ws(
                    "||",
                    *[
                        coalesce(col(c).cast("string"), lit(""))
                        for c in df.columns
                    ]
                ),
                256
            )
        )
    )

## Table Mapping

In [0]:
table_mapping = {

    "Customer.json":
        "bronze_customer",

    "Product.csv":
        "bronze_product",

    "Person.csv":
        "bronze_person",

    "SalesOrderHeader.csv":
        "bronze_sales_order_header",

    "SalesOrderDetail.csv":
        "bronze_sales_order_detail"
}

## Main Ingestion Loop

In [0]:
audit_data = []

files = dbutils.fs.ls(SOURCE_PATH)

for file in files:
    file_name = file.name

    if file_name not in table_mapping:
        continue

    try:
        print("=" * 50)
        print(f"Processing : {file_name}")

        df = read_source_file(file)

        source_count = df.count()

        bronze_df = add_bronze_metadata(
            df,
            file_name
        )

        target_table = table_mapping[file_name]
        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(
                f"{CATALOG}.{SCHEMA}.{target_table}"
            )
        )

        audit_data.append(
            (
                file_name,
                target_table,
                source_count,
                "SUCCESS"
            )
        )
        print(
            f"Loaded {source_count} rows into {target_table}"
        )

    except Exception as e:
        audit_data.append(
            (
                file_name,
                None,
                0,
                f"FAILED : {str(e)}"
            )
        )

        print(str(e))

## Audit Results

In [0]:
audit_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("records_loaded", LongType(), True),
    StructField("status", StringType(), True)
])

audit_df = spark.createDataFrame(
    audit_data,
    audit_schema
)

display(audit_df)

## Save Audit to Table

In [0]:
(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "workspace.final.bronze_ingestion_audit"
    )
)

## Verification

In [0]:
spark.sql("""
SHOW TABLES IN workspace.final
""").display()

### Record Counts

In [0]:
tables = [

    "bronze_customer",
    "bronze_person",
    "bronze_product",
    "bronze_sales_order_header",
    "bronze_sales_order_detail"

]

for table in tables:
    cnt = spark.sql(f"""
        SELECT COUNT(*)
        FROM workspace.final.{table}
    """).collect()[0][0]

    print(f"{table} : {cnt}")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM workspace.final.bronze_customer
    LIMIT 5
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM workspace.final.bronze_product
    LIMIT 5
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM workspace.final.bronze_person
    LIMIT 5
    """)
)

### Summary

In [0]:
summary = []

for table in tables:

    count = spark.sql(f"""
        SELECT COUNT(*)
        FROM workspace.final.{table}
    """).collect()[0][0]

    summary.append((table, count))

summary_df = spark.createDataFrame(
    summary,
    ["table_name", "record_count"]
)

display(summary_df)